<!-- 학습 보강 셀 -->

# 09. Persisting VectorStoreIndex 학습 흐름

이 노트북은 만든 인덱스를 디스크에 저장하고 다시 불러오는 과정을 다룹니다.
실제 RAG 서비스에서는 매번 문서를 다시 임베딩하면 비용과 시간이 크기 때문에 저장/로드가 필수입니다.

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [2]:
# OLLAMA_MODEL_PREP_CELL
# Ollama 모델은 pip/requirements.txt로 설치되지 않습니다.
# 이 셀은 노트북 실행 전에 필요한 로컬 Ollama 모델이 있는지 확인하고, 없으면 자동으로 pull 합니다.
import subprocess

OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_LLM_MODEL = 'gemma2:2b'
OLLAMA_EMBED_MODEL = 'nomic-embed-text'

def _installed_ollama_models() -> set[str]:
    try:
        result = subprocess.run(
            ['ollama', 'list'],
            check=True,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise RuntimeError('Ollama CLI가 설치되어 있지 않습니다. https://ollama.com 에서 설치하세요.') from exc
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Ollama 서버가 실행 중인지 확인하세요. 터미널에서 `ollama serve`를 실행하세요.') from exc

    names = set()
    for line in result.stdout.splitlines()[1:]:
        parts = line.split()
        if parts:
            names.add(parts[0])
    return names

def ensure_ollama_model(model_name: str) -> None:
    installed = _installed_ollama_models()
    candidates = {model_name}
    if ':' not in model_name:
        candidates.add(f'{model_name}:latest')

    if installed.intersection(candidates):
        print(f'이미 설치됨: {model_name}')
        return

    print(f'Ollama 모델 다운로드 중: {model_name}')
    subprocess.run(['ollama', 'pull', model_name], check=True)

for model_name in [OLLAMA_LLM_MODEL, OLLAMA_EMBED_MODEL]:
    ensure_ollama_model(model_name)

이미 설치됨: gemma2:2b
이미 설치됨: nomic-embed-text


In [3]:
# LLM과 임베딩 모델 설정
llm = Ollama(
    model=OLLAMA_LLM_MODEL,
    temperature=0,
    request_timeout=120,
    base_url=OLLAMA_BASE_URL,
)

embed_model = OllamaEmbedding(
    model_name=OLLAMA_EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

In [4]:
# 데이터 로드
# - 저장/로드 예제에서는 pdf_sample2의 논문을 사용합니다.
documents = SimpleDirectoryReader('../NewData/pdf_sample2/').load_data()
print('읽어온 문서 수:', len(documents))

읽어온 문서 수: 11


In [5]:
# 인덱스 생성 및 데이터 임베딩
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model,
    show_progress=True,
)

/Users/cheng80/Documents/WorkSpace/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 12/12 [00:01<00:00,  8.35it/s]


<!-- 학습 보강 셀 -->

## 저장하기 전과 저장한 후의 차이

메모리에 있는 `index`는 현재 커널이 살아 있는 동안만 사용할 수 있습니다.
`persist()`로 저장하면 커널을 재시작해도 같은 인덱스를 다시 로드할 수 있습니다.

In [6]:
# 인덱스 저장
# - storage_context.persist()는 docstore, index_store, vector_store 정보를 디렉토리에 저장합니다.
# - 저장 후에는 문서를 다시 임베딩하지 않고 load_index_from_storage로 불러올 수 있습니다.
persist_dir = './saved_index'
index.storage_context.persist(persist_dir=persist_dir)

<!-- 학습 보강 셀 -->

## 저장 폴더 안의 파일들이 의미하는 것

저장 폴더에는 문서 조각 정보, 인덱스 구조, 벡터 스토어 데이터가 나뉘어 들어갑니다.
파일 하나만 복사해서는 로드가 실패할 수 있으므로, 저장 디렉토리 전체를 하나의 인덱스 묶음으로 다루는 것이 안전합니다.

In [7]:
# 저장된 인덱스 로드 위치 정의
storage_context = StorageContext.from_defaults(persist_dir=persist_dir)

<!-- 학습 보강 셀 -->

## 로드할 때 같은 임베딩 모델을 써야 하는 이유

저장된 벡터는 특정 임베딩 모델이 만든 좌표계 위에 있습니다.
다른 임베딩 모델로 질문 벡터를 만들면 좌표계가 달라져 검색 품질이 크게 떨어질 수 있습니다.
따라서 생성과 로드 시 같은 임베딩 모델을 사용하는 것이 기본 원칙입니다.

In [8]:
# 인덱스 로드
# - 생성할 때 사용한 임베딩 모델과 같은 모델을 지정해야 검색 결과가 일관됩니다.
loaded_index = load_index_from_storage(
    storage_context,
    embed_model=embed_model,
)

2026-06-02 11:32:02,967 - INFO - Loading all indices.


In [9]:
# 로드한 인덱스로 쿼리 엔진 생성
loaded_query_engine = loaded_index.as_query_engine(llm=llm)

2026-06-02 11:32:02,978 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [10]:
# 저장소에서 불러온 인덱스로 질의 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘'
response = loaded_query_engine.query(query)

print()
print('질문:', query)
print('답변:', response)

2026-06-02 11:32:02,999 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:32:05,455 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



질문: 이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘
답변: 이 논문에서는 Transformer 모델이 기존 모델보다 높은 BLEU 점수를 얻을 수 있음을 보여주며,  작은 모델도 큰 모델과 비슷한 성능을 보이는 장점을 제시하고 있습니다. 또한, 이 모델은 다른 모델들에 비해 훨씬 빠른 학습 속도로 학습이 가능합니다. 



#### 메모리 인덱스와 저장소 로드 인덱스 비교

아래 셀은 방금 메모리에 만든 `index`로 같은 질문을 실행해 저장소에서 불러온 결과와 비교하는 용도입니다.

In [11]:
# 메모리에 있는 원본 인덱스로 쿼리 엔진 생성
query_engine = index.as_query_engine(llm=llm)

In [12]:
# 메모리 인덱스로 같은 질문 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘'
response = query_engine.query(query)

print()
print('질문:', query)
print('답변:', response)

2026-06-02 11:32:05,482 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:32:07,982 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



질문: 이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘
답변: 이 논문에서는 Transformer 모델이 기존 모델보다 높은 BLEU 점수를 얻을 수 있음을 보여주며,  작은 모델도 큰 모델과 비슷한 성능을 보이는 장점을 제시하고 있습니다. 또한, 이 모델은 다른 모델들에 비해 훨씬 빠른 학습 속도로 학습이 가능합니다. 

